<a id="titulo"></a>
# ***Asistente experto en analitica de futbol profesional con Gemini, RAG y LangGraph***

Proyecto final de IA Generativa: construccion de un agente experto capaz de responder preguntas sobre analitica de futbol profesional usando una base de conocimiento vectorial propia, Gemini como LLM, Gemini Embeddings, ChromaDB, LangGraph y memoria conversacional.

**Dominio elegido:** scouting y analitica avanzada de jugadores de futbol profesional en las cinco grandes ligas europeas durante la temporada 2024-2025.


<!-- INDICE_GENERADO -->
<a id="indice"></a>
## ***Indice del notebook***

Usa este indice para ir directamente a cada seccion del proyecto.

- [Asistente experto en analitica de futbol profesional con Gemini, RAG y LangGraph](#titulo)
  - [1. Instalacion de dependencias](#sec-1)
  - [2. Imports y configuracion del entorno](#sec-2)
  - [3. Rutas del proyecto](#sec-3)
  - [4. Inicializacion de Gemini](#sec-4)
  - [5. Prueba minima del LLM](#sec-5)
  - [6. Carga inicial del dataset y de los PDFs](#sec-6)
  - [7. Revision rapida de cobertura de datos](#sec-7)
  - [8. Arquitectura MVP del asistente](#sec-8)
  - [9. System prompt personalizado](#sec-9)
    - [9.1 Justificacion del system prompt](#sec-9-1)
  - [10. Preparacion de documentos PDF para RAG](#sec-10)
    - [10.1 Carga de paginas PDF como documentos](#sec-10-1)
  - [11. Chunking de los PDFs](#sec-11)
  - [12. Creacion de la base vectorial con ChromaDB](#sec-12)
    - [12.1 Prueba aislada de embeddings antes de indexar](#sec-12-1)
    - [12.2 Indexacion en ChromaDB](#sec-12-2)
    - [12.3 Recarga de la coleccion existente](#sec-12-3)
  - [13. Prueba del retriever](#sec-13)
  - [14. RAG basico con Gemini](#sec-14)
    - [14.1 Prueba conceptual del RAG](#sec-14-1)
    - [14.2 Inspeccion de documentos usados](#sec-14-2)
    - [14.3 Prueba sobre metricas de porteros](#sec-14-3)
  - [15. Consultas tabulares con pandas](#sec-15)
    - [15.1 Preparacion del DataFrame para consultas](#sec-15-1)
    - [15.2 Buscar jugadores por nombre](#sec-15-2)
    - [15.3 Ranking de jugadores por metrica](#sec-15-3)
    - [15.4 Comparar jugadores](#sec-15-4)
    - [15.5 Convertir resultados pandas a texto para Gemini](#sec-15-5)
    - [15.6 Resultado de la capa pandas](#sec-15-6)
  - [16. Respuesta hibrida: ChromaDB + pandas + Gemini](#sec-16)
    - [16.1 Utilidades para contexto RAG](#sec-16-1)
    - [16.2 Respuesta hibrida para rankings](#sec-16-2)
    - [16.3 Respuesta hibrida para comparaciones](#sec-16-3)
    - [16.4 Inspeccion de fuentes usadas](#sec-16-4)
    - [16.5 Resultado de la respuesta hibrida](#sec-16-5)
  - [17. Agente final con LangGraph y memoria](#sec-17)
    - [17.1 Dise?o del grafo final](#sec-17-1)
    - [17.2 Funciones auxiliares del agente](#sec-17-2)
    - [17.3 Nodos del grafo final](#sec-17-3)
    - [17.4 Construccion del grafo final con MemorySaver](#sec-17-4)
    - [17.5 Visualizacion del grafo final](#sec-17-5)
    - [17.6 Funcion de chat con memoria](#sec-17-6)
    - [17.7 Pruebas del agente final](#sec-17-7)
    - [17.8 Ejemplo de memoria conversacional](#sec-17-8)
    - [17.9 Verificacion de memoria y uso de ChromaDB](#sec-17-9)
  - [18. Interaccion en el notebook y ejemplos documentados](#sec-18)
    - [18.1 Celda interactiva de chat](#sec-18-1)
    - [18.2 Cinco preguntas de ejemplo documentadas](#sec-18-2)
    - [18.3 Documentacion de fuentes usadas en los ejemplos](#sec-18-3)
    - [18.4 Verificacion final de memoria](#sec-18-4)
  - [Resultado de esta fase](#resultado-fase)


<a id="sec-1"></a>
## ***1. Instalacion de dependencias***

Ejecuta esta celda solo si tu entorno no tiene instaladas las librerias. En local se recomienda usar `requirements.txt`; en Google Colab puedes descomentar la instalacion.


In [ ]:
# Esta celda sirve para instalar las librerias si el entorno no las tiene.
# En local normalmente se instalan desde requirements.txt.
# En Google Colab conviene descomentar la linea siguiente y ejecutarla una vez.
!pip install -q langchain langchain-core langchain-community langchain-google-genai langchain-text-splitters langgraph chromadb python-dotenv pypdf pandas

<a id="sec-2"></a>
## ***2. Imports y configuracion del entorno***

Las claves y configuraciones sensibles se cargan desde `.env`. No se debe hardcodear la API key en el notebook.


In [ ]:
# Path permite trabajar con rutas de archivos de forma robusta en Windows, Linux o Colab
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Cargamos el archivo .env para que Python pueda leer GOOGLE_API_KEY y otros parametros
# override=True fuerza a usar el valor actual del .env aunque el kernel tuviera uno antiguo cargado
load_dotenv(override=True)

# Leemos la clave de Gemini desde el .env
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash").strip().strip('"')
GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-001").strip().strip('"')
CHROMA_DB_DIR = os.getenv("CHROMA_DB_DIR", "chroma_db").strip().strip('"')

# Si no hay API key, paramos el notebook con un error claro
if not GOOGLE_API_KEY:
    raise ValueError("No se encontro GOOGLE_API_KEY. Revisa el archivo .env")

# Mostramos informacion de configuracion sin imprimir la API key completa
print("API key cargada:", bool(GOOGLE_API_KEY))
print("Modelo Gemini:", GEMINI_MODEL)
print("Modelo embeddings:", GEMINI_EMBEDDING_MODEL)
print("Directorio Chroma:", CHROMA_DB_DIR)

<a id="sec-3"></a>
## ***3. Rutas del proyecto***

Centralizamos las rutas para que el notebook sea mas mantenible y pueda ejecutarse desde la carpeta `notebooks` o desde la raiz del proyecto.


In [ ]:
# Path.cwd() devuelve la carpeta desde la que se esta ejecutando el notebook
NOTEBOOK_DIR = Path.cwd()

# Si ejecutamos el notebook desde la carpeta notebooks, la raiz del proyecto es el directorio padre
# Si lo ejecutamos desde la raiz del proyecto, PROJECT_ROOT sera la carpeta actual
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# Carpeta donde estan los datos del proyecto
DATA_DIR = PROJECT_ROOT / "data"

# Ruta del CSV con estadisticas de jugadores
CSV_PATH = DATA_DIR / "players_data-2024_2025.csv"

# Lista de PDFs que formaran la base vectorial de conocimiento conceptual
PDF_PATHS = sorted(DATA_DIR.glob("*.pdf"))

# Ruta donde ChromaDB guardara los vectores de forma persistente
CHROMA_PATH = PROJECT_ROOT / CHROMA_DB_DIR

# Comprobamos que el CSV existe
if not CSV_PATH.exists():
    raise FileNotFoundError(f"No se encontro el CSV requerido: {CSV_PATH}")

# Comprobamos que hay al menos 3 PDFs, como pide el enunciado
if len(PDF_PATHS) < 3:
    raise FileNotFoundError("Se necesitan al menos 3 PDFs en la carpeta data para la base de conocimiento.")

# Imprimimos las rutas para verificar que el notebook esta leyendo los archivos correctos
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV:", CSV_PATH)
print("PDFs encontrados:", len(PDF_PATHS))
for pdf_path in PDF_PATHS:
    print("-", pdf_path.name)
print("ChromaDB:", CHROMA_PATH)

<a id="sec-4"></a>
## ***4. Inicializacion de Gemini***

Creamos dos componentes: el modelo conversacional para generar respuestas y el modelo de embeddings para indexar y recuperar informacion desde ChromaDB.


In [ ]:
# Creamos el modelo de lenguaje Gemini para generar respuestas
llm = ChatGoogleGenerativeAI(
    # Modelo definido en .env, por ejemplo gemini-2.5-flash
    model=GEMINI_MODEL,
    # API key cargada desde .env
    google_api_key=GOOGLE_API_KEY,
    # Temperatura baja para respuestas mas consistentes y menos creativas
    temperature=0.2,
)

# Creamos el modelo de embeddings que convertira texto en vectores numericos
embeddings = GoogleGenerativeAIEmbeddings(
    # Modelo de embeddings definido en .env
    model=GEMINI_EMBEDDING_MODEL,
    # API key cargada desde .env
    google_api_key=GOOGLE_API_KEY,
)

# Confirmamos que ambos componentes se han creado
print("LLM inicializado:", GEMINI_MODEL)
print("Embeddings inicializados:", GEMINI_EMBEDDING_MODEL)

<a id="sec-5"></a>
## ***5. Prueba minima del LLM***

Antes de construir el RAG, verificamos que Gemini responde correctamente. Esta prueba no usa todavia la base de conocimiento; solo valida conexion y configuracion.


In [ ]:
# Enviamos una pregunta simple al LLM para comprobar que Gemini responde
respuesta = llm.invoke("Explica en una frase que significa xG en futbol.")

# Imprimimos solo el contenido textual de la respuesta
print(respuesta.content)

<a id="sec-6"></a>
## ***6. Carga inicial del dataset y de los PDFs***

El CSV se usara con pandas para consultar datos exactos de jugadores. Los PDFs se usaran como documentos de conocimiento para ChromaDB y el RAG.


In [ ]:
# Cargamos el CSV de jugadores en un DataFrame de pandas
df = pd.read_csv(CSV_PATH)

# Mostramos dimensiones del dataset: filas y columnas
print("Dataset:", df.shape)

# Mostramos el numero total de columnas
print("Columnas:", len(df.columns))

# Mostramos los PDFs disponibles para la base vectorial
print("Documentos PDF para RAG:")
for pdf_path in PDF_PATHS:
    print("-", pdf_path.name)

# Visualizamos algunas columnas importantes para comprobar que el CSV se cargo bien
df[["Player", "Squad", "Comp", "Pos", "Age", "Min", "xG", "xAG", "PrgC", "PrgP"]].head()

<a id="sec-7"></a>
## ***7. Revision rapida de cobertura de datos***

Comprobamos que el dataset tiene suficiente volumen y variedad para justificar el proyecto: varias ligas, posiciones y miles de registros.


In [ ]:
# Mostramos cuantas filas tiene el dataset
print("Registros:", len(df))

# Contamos cuantos jugadores/registros hay por liga
print("Ligas:")
display(df["Comp"].value_counts())

# Contamos las posiciones mas frecuentes del dataset
print("Posiciones principales:")
display(df["Pos"].value_counts().head(15))

<a id="sec-8"></a>
## ***8. Arquitectura MVP del asistente***

Para mantener el proyecto simple y defendible, usaremos una arquitectura hibrida:

- **ChromaDB + Gemini Embeddings**: para recuperar conocimiento experto desde los 3 PDFs de metricas.
- **pandas + CSV completo**: para consultas numericas exactas sobre todos los jugadores del dataset.
- **Gemini LLM**: para redactar una respuesta clara usando el contexto recuperado y, cuando aplique, los resultados calculados desde el CSV.

Esta decision evita depender de similitud semantica para filtros exactos como `xG > 15`, `Min >= 1000` o `Top 10 por xAG`, que se resuelven mejor con pandas.


<a id="sec-9"></a>
## ***9. System prompt personalizado***

El system prompt define el rol, tono, limites y reglas del asistente. Es una parte obligatoria del enunciado y se documentara tambien en el README.


In [ ]:
# Definimos el system prompt principal del asistente
# Este texto se enviara a Gemini para indicarle como debe comportarse
SYSTEM_PROMPT = """
Eres un asistente experto en analitica de futbol profesional, especializado en scouting, rendimiento de jugadores y metricas avanzadas.

Tu objetivo es ayudar a analizar jugadores de futbol usando dos fuentes de informacion:
1. Contexto recuperado desde ChromaDB, especialmente el glosario de metricas y fichas de scouting indexadas.
2. Resultados calculados desde el CSV con pandas cuando la pregunta requiera rankings, filtros numericos o comparaciones exactas.

Reglas de comportamiento:
- Responde en español, con tono claro, profesional y didactico.
- No inventes datos de jugadores ni valores numericos.
- Si una respuesta requiere datos exactos y no se han proporcionado resultados del CSV, indica que necesitas consultar el dataset.
- Si el contexto recuperado no contiene informacion suficiente, dilo explicitamente.
- Explica las metricas cuando sean importantes para interpretar la respuesta.
- Cuando compares jugadores, ten en cuenta posicion, minutos jugados, liga y rol.
- Prioriza respuestas breves, utiles y basadas en evidencia.

Limitaciones:
- No predigas resultados futuros como si fueran certezas.
- No des recomendaciones de apuestas.
- No afirmes que un jugador es mejor que otro sin explicar con que metricas se justifica.
""".strip()

print(SYSTEM_PROMPT)

<a id="sec-9-1"></a>
### ***9.1 Justificacion del system prompt***

Este prompt se ha diseñado para cumplir el enunciado porque:

- Define un rol experto concreto: analista profesional de futbol.
- Obliga a usar evidencia procedente de ChromaDB y del CSV.
- Reduce alucinaciones al prohibir inventar datos numericos.
- Explica como actuar si falta informacion.
- Mantiene un tono didactico, adecuado para un usuario que quiere entender metricas y decisiones de scouting.

---


<a id="sec-10"></a>
## ***10. Preparacion de documentos PDF para RAG***

Para simplificar el proyecto, la base vectorial se construye solo con los 3 PDFs de metricas. Cada pagina del PDF se convierte en un documento de LangChain con metadatos de fuente y pagina.


In [ ]:
from langchain_core.documents import Document
# RecursiveCharacterTextSplitter divide textos largos en fragmentos mas pequenos.
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma
from pypdf import PdfReader

<a id="sec-10-1"></a>
### ***10.1 Carga de paginas PDF como documentos***

Extraemos el texto de cada pagina. Guardamos metadatos como nombre de archivo y numero de pagina para poder citar la fuente recuperada.


In [ ]:
# Funcion auxiliar para corregir problemas tipicos de codificacion al extraer texto de algunos PDFs
def repair_pdf_text(text):
    # Si aparecen caracteres como m??tricas o f??tbol, suele ser texto UTF-8 interpretado como Latin-1
    if "?" in text or "?" in text:
        try:
            return text.encode("latin1").decode("utf-8")
        except UnicodeError:
            return text
    return text


# Lista donde guardaremos un Document por cada pagina de PDF
pdf_page_documents = []

# Recorremos todos los PDFs encontrados en la carpeta data
for pdf_path in PDF_PATHS:
    # Abrimos el PDF
    reader = PdfReader(str(pdf_path))

    # Recorremos sus paginas.
    for page_number, page in enumerate(reader.pages, start=1):
        # Extraemos texto de la pagina. Si no hay texto, usamos cadena vacia
        text = page.extract_text() or ""

        # Corregimos posibles problemas de codificacion y limpiamos espacios extremos
        text = repair_pdf_text(text).strip()

        # Si una pagina no tiene texto, la saltamos
        if not text:
            continue

        # Creamos un Document con el texto y sus metadatos
        doc = Document(
            page_content=text,
            metadata={
                "source": pdf_path.name,
                "doc_type": "metric_pdf",
                "page": page_number,
            },
        )

        # Añadimos el documento a la lista
        pdf_page_documents.append(doc)

# Mostramos resumen de documentos cargados
print("Paginas PDF convertidas en documentos:", len(pdf_page_documents))
for doc in pdf_page_documents[:3]:
    print("-", doc.metadata, "|", doc.page_content[:120].replace("", " "))

<a id="sec-11"></a>
## ***11. Chunking de los PDFs***

Dividimos las paginas en chunks mas pequenos. Esto mejora la recuperacion porque Chroma puede encontrar fragmentos concretos sobre xG, xA, PSxG, SCA, porteros, defensa, etc.


In [ ]:
# Configuramos el divisor de texto
text_splitter = RecursiveCharacterTextSplitter(
    # Tamano aproximado de cada chunk en caracteres
    chunk_size=900,
    # Solapamiento entre chunks para no perder contexto entre cortes
    chunk_overlap=120,
    # Separadores preferidos: parrafos, lineas, frases y palabras
    separators=["", "", ". ", " ", ""],
)

# Dividimos las paginas de PDF en chunks
pdf_chunks = text_splitter.split_documents(pdf_page_documents)

# Estos chunks son los documentos finales que se indexaran en ChromaDB
rag_documents = pdf_chunks

# Mostramos el resultado del chunking
print("Chunks PDF creados:", len(pdf_chunks))
print("Total documentos RAG:", len(rag_documents))
print(pdf_chunks[0].page_content[:800])
print("Metadatos:", pdf_chunks[0].metadata)

<a id="sec-12"></a>
## ***12. Creacion de la base vectorial con ChromaDB***

Creamos embeddings con Gemini para los chunks de los PDFs y los guardamos en ChromaDB. Esta base vectorial queda dedicada al conocimiento conceptual de metricas futbolisticas.


<a id="sec-12-1"></a>
### ***12.1 Prueba aislada de embeddings antes de indexar***

Antes de enviar documentos a ChromaDB, probamos una sola consulta de embeddings. Si falla, el problema es de modelo, API key o cuota.


In [ ]:
# Probamos el modelo de embeddings con un unico texto corto.
test_embedding = embeddings.embed_query("prueba de embeddings para metricas de futbol")

# Si funciona, deberia devolver una lista de numeros.
print("Dimension del embedding:", len(test_embedding))
print("Primeros valores:", test_embedding[:5])

<a id="sec-12-2"></a>
### ***12.2 Indexacion en ChromaDB***

Usamos una coleccion nueva solo para PDFs. Asi no mezclamos documentos antiguos de jugadores con la nueva arquitectura simplificada.


In [ ]:
"""
# Imports locales para que esta celda funcione aunque se ejecute de forma aislada.
import hashlib
import time

# Nombre de la coleccion dentro de ChromaDB.
COLLECTION_NAME = "asistente_metricas_futbol_pdf"

# Creamos una instancia de ChromaDB usando Gemini Embeddings.
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PATH),
)

# Funcion auxiliar para crear IDs estables a partir de fuente, pagina e indice.
def build_document_id(doc, index):
    raw_id = "|".join([
        str(doc.metadata.get("source", "")),
        str(doc.metadata.get("page", "")),
        str(index),
    ])
    return hashlib.md5(raw_id.encode("utf-8")).hexdigest()

# Lotes pequenos para evitar problemas de cuota.
BATCH_SIZE = 5
SLEEP_SECONDS = 8

# Creamos IDs para todos los chunks.
document_ids = [build_document_id(doc, i) for i, doc in enumerate(rag_documents)]

# Obtenemos IDs ya existentes para no duplicar documentos si reejecutamos la celda.
existing = vectorstore._collection.get(include=[])
existing_ids = set(existing.get("ids", []))

# Insertamos chunks por lotes.
for start in range(0, len(rag_documents), BATCH_SIZE):
    end = start + BATCH_SIZE
    batch_docs = rag_documents[start:end]
    batch_ids = document_ids[start:end]

    pending_pairs = [
        (doc, doc_id)
        for doc, doc_id in zip(batch_docs, batch_ids)
        if doc_id not in existing_ids
    ]

    if not pending_pairs:
        print(f"Lote {start + 1}-{min(end, len(rag_documents))} ya estaba indexado")
        continue

    pending_docs = [pair[0] for pair in pending_pairs]
    pending_ids = [pair[1] for pair in pending_pairs]

    try:
        vectorstore.add_documents(documents=pending_docs, ids=pending_ids)
        existing_ids.update(pending_ids)
        print(f"Indexados documentos {start + 1}-{min(end, len(rag_documents))} de {len(rag_documents)}")
    except Exception as error:
        print("Se detuvo la indexacion por un error, probablemente cuota de Gemini.")
        print("Documentos guardados hasta ahora:", vectorstore._collection.count())
        print("Error:", error)
        break

    if end < len(rag_documents):
        time.sleep(SLEEP_SECONDS)

print("Coleccion creada/cargada:", COLLECTION_NAME)
print("Directorio persistente:", CHROMA_PATH)
print("Documentos en ChromaDB:", vectorstore._collection.count()) 
"""

<a id="sec-12-3"></a>
### ***12.3 Recarga de la coleccion existente***

En futuras ejecuciones, puedes saltar la indexacion y ejecutar solo esta celda para cargar la coleccion ya guardada.


In [ ]:
# Nombre de la coleccion dedicada a los PDFs de metricas
COLLECTION_NAME = "asistente_metricas_futbol_pdf"

# Recargamos la coleccion existente sin recalcular embeddings
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PATH),
)

print("Coleccion cargada:", COLLECTION_NAME)
print("Documentos disponibles:", vectorstore._collection.count())

<a id="sec-13"></a>
## ***13. Prueba del retriever***

Antes de crear el agente, comprobamos que ChromaDB recupera fragmentos relevantes de los PDFs.


In [ ]:
# Convertimos ChromaDB en un retriever de LangChain
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# Funcion auxiliar para probar recuperacion semantica
def probar_retriever(pregunta):
    docs = retriever.invoke(pregunta)
    print("Pregunta:", pregunta)
    print("Documentos recuperados:", len(docs))

    for i, doc in enumerate(docs, start=1):
        print("" + "=" * 80)
        print(f"Documento {i}")
        print("Fuente:", doc.metadata.get("source"))
        print("Pagina:", doc.metadata.get("page"))
        print("Tipo:", doc.metadata.get("doc_type"))
        print(doc.page_content[:900])

    return docs

In [ ]:
docs_prueba_metricas = probar_retriever("Que significa xG y como se interpreta en scouting?")

In [ ]:
docs_prueba_porteros = probar_retriever("Que significa PSxG+/- y como se usa para evaluar porteros?")

<a id="sec-14"></a>
## ***14. RAG basico con Gemini***

Ahora conectamos el retriever con Gemini. Esta version todavia no usa LangGraph ni memoria; sirve para comprobar que el modelo responde usando contexto recuperado desde ChromaDB.


In [ ]:
# Funcion para convertir documentos recuperados en texto legible para el prompt
def format_retrieved_docs(docs):
    # Creamos una lista donde guardaremos cada documento formateado
    formatted = []

    # Recorremos los documentos recuperados por ChromaDB
    for i, doc in enumerate(docs, start=1):
        # Extraemos metadatos utiles
        source = doc.metadata.get("source", "fuente desconocida")
        doc_type = doc.metadata.get("doc_type", "tipo desconocido")
        page = doc.metadata.get("page")

        # Creamos una cabecera para saber de donde viene cada fragmento
        header = f"[Fragmento recuperado {i} | fuente: {source} | tipo: {doc_type}"
        if page:
            header += f" | pagina: {page}"
        header += "]"

        # Juntamos cabecera y contenido del documento
        formatted.append(header + "" + doc.page_content)

    # Devolvemos todos los documentos separados por una linea visual
    return "---".join(formatted)


# Funcion principal de RAG basico
def responder_con_rag_basico(pregunta, k=5):
    # Recuperamos documentos relevantes desde ChromaDB
    docs = retriever.invoke(pregunta)

    # Nos quedamos con los k primeros por si el retriever devuelve mas
    docs = docs[:k]

    # Convertimos los documentos recuperados en texto para Gemini
    contexto = format_retrieved_docs(docs)

    # Creamos el prompt final con instrucciones, pregunta y contexto
    prompt = f"""
{SYSTEM_PROMPT}

Pregunta del usuario:
{pregunta}

Contexto recuperado desde ChromaDB:
{contexto}

Instrucciones para responder:
- Responde usando el contexto recuperado.
- Si el contexto no contiene la informacion necesaria, dilo claramente.
- Cita brevemente las fuentes usando el nombre del PDF y la pagina indicada en los metadatos.
""".strip()

    # Enviamos el prompt a Gemini
    respuesta = llm.invoke(prompt)

    # Devolvemos respuesta y documentos para poder inspeccionar el RAG
    return {
        "respuesta": respuesta.content,
        "documentos": docs,
        "prompt": prompt,
    }

<a id="sec-14-1"></a>
### ***14.1 Prueba conceptual del RAG***

Esta prueba debe recuperar fragmentos del glosario y responder usando definiciones de metricas.


In [ ]:
resultado_xg = responder_con_rag_basico("Que significa xG y como se interpreta en scouting?")
print(resultado_xg["respuesta"])

<a id="sec-14-2"></a>
### ***14.2 Inspeccion de documentos usados***

Esta celda permite demostrar que la respuesta no sale solo del LLM, sino de documentos recuperados por ChromaDB.


In [ ]:
for i, doc in enumerate(resultado_xg["documentos"], start=1):
    print("=" * 80)
    print(f"Documento recuperado {i}")
    print("Metadatos:", doc.metadata)
    print(doc.page_content[:800])

<a id="sec-14-3"></a>
### ***14.3 Prueba sobre metricas de porteros***

Como ChromaDB ahora solo contiene PDFs conceptuales, esta prueba valida una pregunta sobre metricas de porteros. Los jugadores concretos se consultaran despues con pandas.


In [ ]:
resultado_porteros = responder_con_rag_basico("Explica PSxG, PSxG/SoT y PSxG+/- para evaluar porteros")
print(resultado_porteros["respuesta"])

---


<a id="sec-15"></a>
## ***15. Consultas tabulares con pandas***

ChromaDB se usa para explicar metricas desde los PDFs. Para consultar jugadores, rankings y filtros numericos usamos pandas sobre el CSV completo, porque es mas fiable para operaciones exactas.


<a id="sec-15-1"></a>
### ***15.1 Preparacion del DataFrame para consultas***

Convertimos columnas numericas importantes y definimos una seleccion de metricas utiles para mostrar resultados de scouting.


In [ ]:
# Creamos una copia de trabajo para consultas tabulares
df_stats = df.copy()

# Columnas numericas frecuentes en las consultas del asistente
NUMERIC_COLUMNS = [
    "Age", "Born", "MP", "Starts", "Min", "90s",
    "Gls", "Ast", "G+A", "G-PK", "PK", "PKatt",
    "xG", "npxG", "xAG", "xA", "npxG+xAG", "xG+xAG",
    "Sh", "SoT", "SoT%", "Sh/90", "SoT/90", "G/Sh", "G/SoT", "Dist", "npxG/Sh", "G-xG", "np:G-xG",
    "Cmp", "Att", "Cmp%", "TotDist", "PrgDist", "KP", "1/3", "PPA", "CrsPA", "PrgP",
    "SCA", "SCA90", "GCA", "GCA90",
    "Tkl", "TklW", "Tkl%", "Blocks", "Int", "Tkl+Int", "Clr", "Err",
    "Touches", "Att Pen", "Carries", "PrgC", "CPA", "Succ", "Succ%", "Mis", "Dis", "Rec", "PrgR",
    "CrdY", "CrdR", "Fls", "Fld", "Off", "Recov", "Won", "Won%",
    "GA", "GA90", "SoTA", "Saves", "Save%", "CS", "CS%", "PSxG", "PSxG/SoT", "PSxG+/-", "#OPA", "#OPA/90", "AvgDist"
]

# Convertimos a numericas solo las columnas que existan en el dataset
for col in NUMERIC_COLUMNS:
    if col in df_stats.columns:
        df_stats[col] = pd.to_numeric(df_stats[col], errors="coerce")

# Columnas basicas para identificar jugadores en resultados
ID_COLUMNS = ["Player", "Nation", "Pos", "Squad", "Comp", "Age", "Min"]

# Metricas por defecto para mostrar fichas resumidas
DEFAULT_SCOUTING_COLUMNS = [
    "Player", "Squad", "Comp", "Pos", "Age", "Min",
    "Gls", "Ast", "xG", "npxG", "xAG", "xA",
    "PrgC", "PrgP", "PrgR", "SCA", "SCA90",
    "Tkl", "Int", "Tkl+Int",
    "GA", "Save%", "PSxG", "PSxG+/-"
]

# Nos quedamos solo con columnas existentes para evitar errores si alguna no aparece
DEFAULT_SCOUTING_COLUMNS = [col for col in DEFAULT_SCOUTING_COLUMNS if col in df_stats.columns]

print("DataFrame preparado:", df_stats.shape)
print("Columnas numericas convertidas:", sum(col in df_stats.columns for col in NUMERIC_COLUMNS))

<a id="sec-15-2"></a>
### ***15.2 Buscar jugadores por nombre***

Esta funcion permite encontrar jugadores en todo el CSV, aunque no esten indexados en ChromaDB.


In [ ]:
# Busca jugadores cuyo nombre contenga el texto indicado
def buscar_jugador(nombre, max_resultados=10):
    # Normalizamos la consulta a minusculas
    nombre = nombre.lower().strip()

    # Filtramos filas donde Player contenga el texto buscado
    resultados = df_stats[df_stats["Player"].str.lower().str.contains(nombre, na=False)].copy()

    # Ordenamos por minutos para mostrar primero registros con mas muestra
    resultados = resultados.sort_values("Min", ascending=False)

    # Devolvemos columnas utiles y limitamos el numero de resultados
    return resultados[DEFAULT_SCOUTING_COLUMNS].head(max_resultados)

In [ ]:
# Prueba: buscar un jugador concreto en el CSV completo
buscar_jugador("Salah")

<a id="sec-15-3"></a>
### ***15.3 Ranking de jugadores por metrica***

Esta funcion crea rankings exactos usando pandas. Es mejor que ChromaDB para preguntas como top 10 por xG, xAG, PrgP o PSxG+/-.


In [ ]:
# Crea un ranking de jugadores por una metrica numerica
def top_jugadores_por_metrica(metrica, n=10, min_minutos=900, posicion=None, liga=None, ascendente=False):
    # Validamos que la metrica exista
    if metrica not in df_stats.columns:
        raise ValueError(f"La metrica '{metrica}' no existe en el dataset.")

    # Validamos que la metrica sea numerica o convertible
    if not pd.api.types.is_numeric_dtype(df_stats[metrica]):
        raise ValueError(f"La metrica '{metrica}' no es numerica.")

    # Partimos del dataset completo
    datos = df_stats.copy()

    # Aplicamos filtro minimo de minutos si existe la columna Min
    if "Min" in datos.columns:
        datos = datos[datos["Min"].fillna(0) >= min_minutos]

    # Filtramos por posicion si se indica. Usa contains para capturar posiciones combinadas como FW,MF
    if posicion:
        datos = datos[datos["Pos"].str.contains(posicion, case=False, na=False)]

    # Filtramos por liga si se indica
    if liga:
        datos = datos[datos["Comp"].str.contains(liga, case=False, na=False)]

    # Quitamos filas sin valor en la metrica
    datos = datos.dropna(subset=[metrica])

    # Ordenamos por la metrica
    datos = datos.sort_values(metrica, ascending=ascendente)

    # Seleccionamos columnas de salida
    output_cols = [col for col in ID_COLUMNS + [metrica] if col in datos.columns]

    # Devolvemos el top n
    return datos[output_cols].head(n)

In [ ]:
# Prueba: top 10 jugadores por xG con al menos 900 minutos
top_jugadores_por_metrica("xG", n=10, min_minutos=900)

<a id="sec-15-4"></a>
### ***15.4 Comparar jugadores***

Esta funcion compara dos jugadores usando un conjunto pequeno de metricas interpretables. Si hay varios registros para un jugador, toma el de mas minutos.


In [ ]:
# Devuelve la fila mas representativa de un jugador, usando el registro con mas minutos
def obtener_fila_jugador(nombre):
    # Buscamos coincidencias por nombre
    coincidencias = df_stats[df_stats["Player"].str.lower().str.contains(nombre.lower().strip(), na=False)].copy()

    # Si no hay coincidencias, devolvemos None
    if coincidencias.empty:
        return None

    # Ordenamos por minutos y devolvemos la primera fila
    coincidencias = coincidencias.sort_values("Min", ascending=False)
    return coincidencias.iloc[0]


# Compara dos jugadores en una lista de metricas.
def comparar_jugadores(jugador_1, jugador_2, metricas=None):
    # Metricas por defecto para comparaciones generales
    if metricas is None:
        metricas = ["Min", "Gls", "Ast", "xG", "npxG", "xAG", "xA", "PrgC", "PrgP", "PrgR", "SCA", "SCA90", "Tkl", "Int", "PSxG+/-"]

    # Obtenemos la fila principal de cada jugador
    fila_1 = obtener_fila_jugador(jugador_1)
    fila_2 = obtener_fila_jugador(jugador_2)

    # Controlamos si falta alguno
    if fila_1 is None or fila_2 is None:
        faltan = []
        if fila_1 is None:
            faltan.append(jugador_1)
        if fila_2 is None:
            faltan.append(jugador_2)
        raise ValueError("No se encontraron jugadores: " + ", ".join(faltan))

    # Nos quedamos solo con metricas existentes
    metricas = [m for m in metricas if m in df_stats.columns]

    # Construimos una tabla comparativa
    filas = []
    for fila in [fila_1, fila_2]:
        registro = {
            "Player": fila.get("Player"),
            "Squad": fila.get("Squad"),
            "Comp": fila.get("Comp"),
            "Pos": fila.get("Pos"),
            "Age": fila.get("Age"),
        }
        for metrica in metricas:
            registro[metrica] = fila.get(metrica)
        filas.append(registro)

    return pd.DataFrame(filas)

In [ ]:
# Prueba: comparar dos atacantes
comparar_jugadores("Mohamed Salah", "Raphinha")

<a id="sec-15-5"></a>
### ***15.5 Convertir resultados pandas a texto para Gemini***

Gemini no lee DataFrames directamente. Convertimos los resultados tabulares a texto para que pueda redactar una explicacion clara.


In [ ]:
!pip install tabulate

In [ ]:
# Convierte un DataFrame pequeno en texto legible para incluirlo en un prompt
def dataframe_a_texto(df_resultado, titulo="Resultados del CSV"):
    # Si no hay resultados, devolvemos mensaje claro
    if df_resultado is None or df_resultado.empty:
        return f"{titulo}: no se encontraron resultados."

    # Limitamos a 15 filas para no crear prompts demasiado largos
    df_mostrar = df_resultado.head(15).copy()

    # Convertimos el DataFrame a texto tipo tabla Markdown
    tabla = df_mostrar.to_markdown(index=False)

    return f"{titulo}:{tabla}"


# Ejemplo de conversion a texto
ejemplo_top_xg = top_jugadores_por_metrica("xG", n=5, min_minutos=900)
print(dataframe_a_texto(ejemplo_top_xg, "Top 5 por xG"))

<a id="sec-15-6"></a>
### ***15.6 Resultado de la capa pandas***

Con estas funciones, el asistente ya puede consultar el CSV completo sin depender de ChromaDB para datos numericos. El siguiente paso sera combinar estos resultados con el contexto RAG de los PDFs y generar una respuesta final con Gemini.


<a id="sec-16"></a>
## ***16. Respuesta hibrida: ChromaDB + pandas + Gemini***

En esta fase combinamos las dos fuentes del asistente:

- **pandas** consulta datos exactos del CSV completo.
- **ChromaDB** recupera explicaciones conceptuales desde los PDFs.
- **Gemini** redacta una respuesta final usando ambos contextos.

Esto permite responder preguntas mixtas como: "?Qu? jugadores tienen m?s xG y qu? significa xG?".


<a id="sec-16-1"></a>
### ***16.1 Utilidades para contexto RAG***

Reutilizamos el retriever para recuperar fragmentos relevantes de los PDFs, pero devolvemos el contexto como texto para incluirlo en el prompt final.


In [ ]:
# Recupera contexto conceptual desde ChromaDB y lo devuelve como texto
def recuperar_contexto_metricas(pregunta, k=4):
    # Recuperamos fragmentos relevantes de los PDFs
    docs = retriever.invoke(pregunta)[:k]

    # Convertimos los documentos recuperados a texto usando la funcion creada en el RAG basico
    contexto = format_retrieved_docs(docs)

    # Devolvemos contexto y documentos para poder inspeccionar fuentes
    return contexto, docs

<a id="sec-16-2"></a>
### ***16.2 Respuesta hibrida para rankings***

Esta funcion sirve para preguntas como:

- ?Qu? jugadores tienen m?s xG y qu? significa xG?
- Dame el top 10 por xAG y explica c?mo interpretar la m?trica.
- ?Qu? porteros lideran en PSxG+/- y qu? significa?


In [ ]:
# Responde una pregunta de ranking combinando pandas + ChromaDB + Gemini
def responder_ranking_hibrido(metrica, n=10, min_minutos=900, posicion=None, liga=None, ascendente=False):
    # 1. pandas calcula el ranking exacto desde el CSV completo
    ranking_df = top_jugadores_por_metrica(
        metrica=metrica,
        n=n,
        min_minutos=min_minutos,
        posicion=posicion,
        liga=liga,
        ascendente=ascendente,
    )

    # 2. Convertimos el resultado tabular a texto para Gemini
    texto_csv = dataframe_a_texto(
        ranking_df,
        titulo=f"Top {n} jugadores por {metrica} con minimo {min_minutos} minutos"
    )

    # 3. ChromaDB recupera explicacion conceptual de la metrica
    pregunta_contexto = f"Que significa {metrica} y como se interpreta en analitica de futbol?"
    contexto_rag, docs = recuperar_contexto_metricas(pregunta_contexto, k=4)

    # 4. Construimos el prompt final con system prompt, datos del CSV y contexto RAG
    prompt = f"""
{SYSTEM_PROMPT}

Tarea:
Genera una respuesta de analisis para un ranking de jugadores por la metrica {metrica}.

Datos calculados desde el CSV completo con pandas:
{texto_csv}

Contexto conceptual recuperado desde ChromaDB:
{contexto_rag}

Instrucciones:
- Explica brevemente que significa la metrica {metrica}.
- Presenta el ranking de forma clara.
- Interpreta los resultados sin inventar datos adicionales.
- Recuerda que el filtro minimo de minutos es {min_minutos}.
- Cita las fuentes conceptuales usando el PDF y la pagina cuando sea relevante.
""".strip()

    # 5. Gemini genera la respuesta final
    respuesta = llm.invoke(prompt)

    # 6. Devolvemos todo para inspeccion
    return {
        "respuesta": respuesta.content,
        "tabla": ranking_df,
        "documentos": docs,
        "prompt": prompt,
    }

In [ ]:
# Prueba hibrida: ranking por xG + explicacion de xG desde PDFs
resultado_ranking_xg = responder_ranking_hibrido("xG", n=5, min_minutos=900)
print(resultado_ranking_xg["respuesta"])

<a id="sec-16-3"></a>
### ***16.3 Respuesta hibrida para comparaciones***

Esta funcion sirve para preguntas como:

- Compara a Mohamed Salah y Raphinha usando xG, xAG, PrgC y SCA.
- Compara dos porteros usando Save%, PSxG y PSxG+/-.


In [ ]:
# Responde una comparacion de dos jugadores combinando pandas + ChromaDB + Gemini
def responder_comparacion_hibrida(jugador_1, jugador_2, metricas=None):
    # 1. Si no se pasan metricas, usamos una seleccion general
    if metricas is None:
        metricas = ["Min", "Gls", "Ast", "xG", "npxG", "xAG", "xA", "PrgC", "PrgP", "PrgR", "SCA", "SCA90"]

    # 2. pandas obtiene la tabla comparativa desde el CSV completo
    comparacion_df = comparar_jugadores(jugador_1, jugador_2, metricas=metricas)

    # 3. Convertimos la tabla a texto para Gemini
    texto_csv = dataframe_a_texto(
        comparacion_df,
        titulo=f"Comparacion entre {jugador_1} y {jugador_2}"
    )

    # 4. Recuperamos contexto conceptual sobre las metricas usadas
    metricas_texto = ", ".join(metricas)
    pregunta_contexto = f"Explica como interpretar estas metricas de futbol: {metricas_texto}"
    contexto_rag, docs = recuperar_contexto_metricas(pregunta_contexto, k=5)

    # 5. Construimos prompt final
    prompt = f"""
{SYSTEM_PROMPT}

Tarea:
Compara a {jugador_1} y {jugador_2} usando las metricas indicadas.

Datos calculados desde el CSV completo con pandas:
{texto_csv}

Contexto conceptual recuperado desde ChromaDB:
{contexto_rag}

Instrucciones:
- Compara solo con los datos proporcionados.
- Explica las metricas mas importantes para entender la comparacion.
- Ten en cuenta minutos, posicion y liga.
- No declares un ganador absoluto si las metricas muestran perfiles distintos.
- Cita las fuentes conceptuales usando el PDF y la pagina cuando sea relevante.
""".strip()

    # 6. Gemini genera respuesta
    respuesta = llm.invoke(prompt)

    # 7. Devolvemos respuesta, tabla y documentos
    return {
        "respuesta": respuesta.content,
        "tabla": comparacion_df,
        "documentos": docs,
        "prompt": prompt,
    }

In [ ]:
# Prueba hibrida: comparacion de dos jugadores usando datos del CSV y explicaciones desde PDFs
resultado_comparacion = responder_comparacion_hibrida(
    "Mohamed Salah",
    "Raphinha",
    metricas=["Min", "Gls", "Ast", "xG", "npxG", "xAG", "xA", "PrgC", "PrgP", "PrgR", "SCA", "SCA90"]
)
print(resultado_comparacion["respuesta"])

<a id="sec-16-4"></a>
### ***16.4 Inspeccion de fuentes usadas***

Estas celdas ayudan a demostrar que la respuesta hibrida usa pandas para datos y ChromaDB para contexto conceptual.


In [ ]:
# Tabla exacta calculada con pandas para el ranking
display(resultado_ranking_xg["tabla"])

# Documentos conceptuales recuperados desde ChromaDB
for i, doc in enumerate(resultado_ranking_xg["documentos"], start=1):
    print("=" * 80)
    print(f"Fuente {i}:", doc.metadata)
    print(doc.page_content[:600])

<a id="sec-16-5"></a>
### ***16.5 Resultado de la respuesta hibrida***

Con estas funciones ya podemos responder preguntas que combinan datos exactos del CSV y explicaciones conceptuales de los PDFs. Esta sera la base que luego envolveremos en LangGraph.


<a id="sec-17"></a>
## ***17. Agente final con LangGraph y memoria***

En esta seccion construimos el agente final del proyecto. Para simplificar la entrega, dejamos un unico agente que ya incluye:

- LangGraph para organizar el flujo.
- ChromaDB para recuperar contexto de los PDFs de metricas.
- pandas para consultar el CSV completo cuando hay rankings o comparaciones.
- Gemini para generar la respuesta final.
- `HumanMessage` y `AIMessage` para representar el historial conversacional.
- `MemorySaver` para compilar el grafo con memoria/checkpointing por `thread_id`.


<a id="sec-17-1"></a>
### ***17.1 Dise?o del grafo final***

El grafo final tiene un flujo lineal y facil de explicar:

1. `clasificar_pregunta`: detecta si la pregunta es conceptual, ranking o comparacion.
2. `recuperar_contexto`: recupera siempre contexto desde ChromaDB.
3. `consultar_csv`: usa pandas si la pregunta requiere datos de jugadores.
4. `generar_respuesta`: genera la respuesta final con Gemini usando contexto, datos e historial.

Este dise?o asegura que el agente usa la base vectorial en cada consulta.


In [ ]:
from typing import TypedDict, Optional, List, Dict, Any
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import re


# Estado final del agente con memoria conversacional
class AgentState(TypedDict):
    # Pregunta actual del usuario
    pregunta: str
    # Historial conversacional formateado como texto
    historial: str
    # Tipo de consulta detectada: conceptual, ranking o comparacion
    tipo_consulta: Optional[str]
    # Contexto textual recuperado desde ChromaDB
    contexto_rag: Optional[str]
    # Metadatos de documentos recuperados desde ChromaDB
    fuentes_rag: Optional[List[Dict[str, Any]]]
    # Resultados calculados desde el CSV, convertidos a texto
    datos_csv: Optional[str]
    # Respuesta final generada por Gemini
    respuesta: Optional[str]

<a id="sec-17-2"></a>
### ***17.2 Funciones auxiliares del agente***

Estas funciones mantienen el agente simple. Detectan el tipo de pregunta, la metrica principal y los nombres de jugadores en comparaciones sencillas.


In [ ]:
# Lista corta de metricas que el agente puede detectar si el usuario escribe el nombre exacto.
METRICAS_DETECTABLES = [
    "PSxG+/-", "PSxG/SoT", "Save%", "SoT%", "Sh/90", "SoT/90", "SCA90", "GCA90",
    "xG", "npxG", "xAG", "xA", "Gls", "Ast",
    "PrgC", "PrgP", "PrgR", "SCA", "GCA",
    "PSxG", "GA90", "Min", "Cmp%", "Tkl+Int", "Won%"
]

# Mapeo de expresiones naturales en espanol a columnas reales del CSV.
MAPEO_METRICAS = {
    # Produccion ofensiva
    "goles esperados sin penalti": "npxG",
    "goles esperados sin penaltis": "npxG",
    "goles esperados": "xG",
    "expected goals": "xG",
    "asistencias esperadas": "xAG",
    "expected assists": "xA",
    "asistencias": "Ast",
    "goles": "Gls",
    "goles y asistencias": "G+A",
    "maximo goleador": "Gls",
    "máximo goleador": "Gls",
    "mejor goleador": "Gls",
    "maximo asistente": "Ast",
    "máximo asistente": "Ast",
    "mejor asistente": "Ast",

    # Tiro y precision
    "precision de tiro": "SoT%",
    "precisión de tiro": "SoT%",
    "porcentaje de tiros a puerta": "SoT%",
    "tiros a puerta": "SoT",
    "tiros": "Sh",
    "remates": "Sh",
    "remates a puerta": "SoT",
    "tiros por 90": "Sh/90",
    "tiros a puerta por 90": "SoT/90",
    "calidad media de tiro": "npxG/Sh",
    "distancia de tiro": "Dist",
    "eficacia de finalizacion": "G/Sh",
    "eficacia de finalización": "G/Sh",

    # Pase y creacion
    "pases": "Att",
    "pases intentados": "Att",
    "pases completados": "Cmp",
    "volumen de pase": "Att",
    "mayor volumen de pase": "Att",
    "participacion en pase": "Att",
    "participación en pase": "Att",
    "pases progresivos": "PrgP",
    "conducciones progresivas": "PrgC",
    "recepciones progresivas": "PrgR",
    "pases clave": "KP",
    "pases al area": "PPA",
    "pases al área": "PPA",
    "centros al area": "CrsPA",
    "centros al área": "CrsPA",
    "acciones creadoras de tiro": "SCA",
    "creacion de tiro": "SCA",
    "creación de tiro": "SCA",
    "sca por 90": "SCA90",
    "acciones creadoras de gol": "GCA",
    "gca por 90": "GCA90",
    "toques": "Touches",
    "toques de balon": "Touches",
    "toques de bal?n": "Touches",
    "regates": "Succ",
    "regates exitosos": "Succ",
    "porcentaje de regate": "Succ%",
    "eficacia en regate": "Succ%",
    "perdidas": "Dis",
    "pérdidas": "Dis",
    "controles fallidos": "Mis",
    "porcentaje de pase": "Cmp%",
    "precision de pase": "Cmp%",
    "precisión de pase": "Cmp%",

    # Defensa
    "entradas": "Tkl",
    "entradas ganadas": "TklW",
    "intercepciones": "Int",
    "bloqueos": "Blocks",
    "despejes": "Clr",
    "errores": "Err",
    "recuperaciones": "Recov",
    "faltas cometidas": "Fls",
    "faltas recibidas": "Fld",
    "tarjetas amarillas": "CrdY",
    "tarjetas rojas": "CrdR",
    "duelos ganados": "Won",
    "duelos aereos": "Won%",
    "duelos aéreos": "Won%",
    "actividad defensiva": "Tkl+Int",

    # Porteros
    "porcentaje de paradas": "Save%",
    "paradas": "Saves",
    "goles encajados por 90": "GA90",
    "goles recibidos": "GA",
    "goles encajados": "GA",
    "tiros a puerta recibidos": "SoTA",
    "porterias a cero": "CS",
    "porterías a cero": "CS",
    "clean sheets": "CS",
    "post shot xg": "PSxG",
    "post-shot xg": "PSxG",
    "psxg menos goles": "PSxG+/-",
    "paradas sobre lo esperado": "PSxG+/-",
    "rendimiento de parada": "PSxG+/-",
    "portero libero": "#OPA/90",
    "portero líbero": "#OPA/90",

    # Disponibilidad
    "minutos": "Min",
    "partidos": "MP",
    "titularidades": "Starts",
    "partidos de titular": "Starts",
    "edad": "Age",
    "jugador mas joven": "Age",
    "jugador más joven": "Age",
    "jugador mas veterano": "Age",
    "jugador más veterano": "Age",
}

# Mapeo de posiciones en espanol a codigos del CSV.
MAPEO_POSICIONES = {
    "delantero": "FW",
    "delanteros": "FW",
    "atacante": "FW",
    "atacantes": "FW",
    "extremo": "FW",
    "extremos": "FW",
    "mediapunta": "FW",
    "mediapuntas": "FW",
    "centrocampista": "MF",
    "centrocampistas": "MF",
    "mediocentro": "MF",
    "mediocentros": "MF",
    "medio": "MF",
    "medios": "MF",
    "defensa": "DF",
    "defensas": "DF",
    "central": "DF",
    "centrales": "DF",
    "lateral": "DF",
    "laterales": "DF",
    "portero": "GK",
    "porteros": "GK",
    "arquero": "GK",
    "arqueros": "GK",
    "guardameta": "GK",
    "guardametas": "GK",
}

# Mapeo de ligas en lenguaje natural a valores aproximados del CSV.
MAPEO_LIGAS = {
    "premier": "Premier League",
    "premier league": "Premier League",
    "liga inglesa": "Premier League",
    "la liga": "La Liga",
    "liga espa?ola": "La Liga",
    "serie a": "Serie A",
    "liga italiana": "Serie A",
    "bundesliga": "Bundesliga",
    "liga alemana": "Bundesliga",
    "ligue 1": "Ligue 1",
    "liga francesa": "Ligue 1",
}


# Clasifica una pregunta en conceptual, ranking o comparacion.
def clasificar_tipo_pregunta(pregunta):
    # Pasamos la pregunta a minusculas para buscar palabras clave.
    pregunta_lower = pregunta.lower()

    # Si aparecen palabras de comparacion, la tratamos como comparacion.
    if any(palabra in pregunta_lower for palabra in ["compara", "comparar", " vs ", " versus "]):
        return "comparacion"

    # Si aparecen palabras de ranking, la tratamos como ranking.
    if any(palabra in pregunta_lower for palabra in ["top", "mejores", "mayor", "mayores", "mas ", "m?s ", "lideran"]):
        return "ranking"

    # Si no encaja en lo anterior, sera conceptual.
    return "conceptual"


# Detecta una metrica dentro de la pregunta.
def detectar_metrica_en_pregunta(pregunta, metrica_por_defecto="xG"):
    # Pasamos la pregunta a minusculas.
    pregunta_lower = pregunta.lower()

    # Primero buscamos expresiones naturales como "precision de tiro".
    for expresion, metrica in MAPEO_METRICAS.items():
        if expresion in pregunta_lower:
            return metrica

    # Despues buscamos nombres exactos de columnas como xG o PSxG+/-.
    for metrica in METRICAS_DETECTABLES:
        if metrica.lower() in pregunta_lower:
            return metrica

    # Si no detecta ninguna, usamos xG como valor por defecto.
    return metrica_por_defecto


# Detecta una posicion mencionada en lenguaje natural.
def detectar_posicion_en_pregunta(pregunta):
    # Pasamos la pregunta a minusculas.
    pregunta_lower = pregunta.lower()

    # Buscamos sinonimos de posicion.
    for expresion, posicion in MAPEO_POSICIONES.items():
        if expresion in pregunta_lower:
            return posicion

    # Si no aparece posicion, no filtramos.
    return None


# Detecta una liga mencionada en lenguaje natural.
def detectar_liga_en_pregunta(pregunta):
    # Pasamos la pregunta a minusculas.
    pregunta_lower = pregunta.lower()

    # Buscamos sinonimos de liga.
    for expresion, liga in MAPEO_LIGAS.items():
        if expresion in pregunta_lower:
            return liga

    # Si no aparece liga, no filtramos.
    return None


# Detecta si el usuario pide orden ascendente, util para metricas donde menor es mejor.
def detectar_orden_ascendente(pregunta):
    pregunta_lower = pregunta.lower()
    return any(expresion in pregunta_lower for expresion in ["menos", "menor", "menores", "mas bajo", "m?s bajo", "peor"])


# Extrae dos nombres de una pregunta de comparacion simple.
def extraer_jugadores_comparacion(pregunta):
    # Patron para preguntas tipo: "Compara a Mohamed Salah y Raphinha usando xG".
    patron = r"compara(?:r)?\s+a?\s*(.*?)\s+y\s+(.*?)(?:\s+usando|\s+con|\s+en|$)"
    match = re.search(patron, pregunta, flags=re.IGNORECASE)

    # Si el patron funciona, devolvemos los dos nombres limpios.
    if match:
        jugador_1 = match.group(1).strip(" .,;:")
        jugador_2 = match.group(2).strip(" .,;:")
        return jugador_1, jugador_2

    # Patron alternativo para preguntas tipo: "Mohamed Salah vs Raphinha".
    if " vs " in pregunta.lower():
        partes = re.split(r"\s+vs\s+", pregunta, flags=re.IGNORECASE)
        if len(partes) >= 2:
            return partes[0].strip(" .,;:"), partes[1].strip(" .,;:")

    # Si no se puede extraer, devolvemos None.
    return None, None

<a id="sec-17-3"></a>
### ***17.3 Nodos del grafo final***

Cada nodo recibe el estado, a?ade informacion y devuelve solo los campos actualizados. Esta separacion facilita explicar y depurar el agente.


In [ ]:
# Nodo 1: clasifica la pregunta.
def nodo_clasificar_pregunta(state: AgentState):
    # Leemos la pregunta del estado.
    pregunta = state["pregunta"]

    # Clasificamos el tipo de consulta.
    tipo = clasificar_tipo_pregunta(pregunta)

    # Devolvemos solo el campo actualizado.
    return {"tipo_consulta": tipo}


# Nodo 2: recupera siempre contexto desde ChromaDB.
def nodo_recuperar_contexto(state: AgentState):
    # Leemos la pregunta original.
    pregunta = state["pregunta"]

    # Recuperamos contexto conceptual desde los PDFs indexados.
    contexto, docs = recuperar_contexto_metricas(pregunta, k=4)

    # Guardamos metadatos de fuentes para demostrar uso de ChromaDB.
    fuentes = [doc.metadata for doc in docs]

    # Devolvemos contexto y fuentes.
    return {
        "contexto_rag": contexto,
        "fuentes_rag": fuentes,
    }


# Nodo 3: consulta el CSV con pandas si aplica.
def nodo_consultar_csv(state: AgentState):
    # Leemos pregunta y tipo de consulta.
    pregunta = state["pregunta"]
    tipo = state["tipo_consulta"]

    # Caso ranking: detectamos metrica, posicion, liga y orden, y calculamos top con pandas.
    if tipo == "ranking":
        metrica = detectar_metrica_en_pregunta(pregunta)
        posicion = detectar_posicion_en_pregunta(pregunta)
        liga = detectar_liga_en_pregunta(pregunta)
        ascendente = detectar_orden_ascendente(pregunta)

        ranking_df = top_jugadores_por_metrica(
            metrica=metrica,
            n=10,
            min_minutos=900,
            posicion=posicion,
            liga=liga,
            ascendente=ascendente,
        )

        descripcion_filtros = []
        if posicion:
            descripcion_filtros.append(f"posicion {posicion}")
        if liga:
            descripcion_filtros.append(f"liga {liga}")
        descripcion_filtros = ", ".join(descripcion_filtros) if descripcion_filtros else "sin filtro de posicion/liga"

        datos_csv = dataframe_a_texto(
            ranking_df,
            f"Top 10 por {metrica} ({descripcion_filtros}, minimo 900 minutos)"
        )
        return {"datos_csv": datos_csv}

    # Caso comparacion: intentamos extraer dos jugadores y compararlos.
    if tipo == "comparacion":
        jugador_1, jugador_2 = extraer_jugadores_comparacion(pregunta)

        # Si no hemos podido extraerlos, dejamos un mensaje claro.
        if not jugador_1 or not jugador_2:
            return {"datos_csv": "No se pudieron identificar claramente los dos jugadores a comparar."}

        # Metricas sencillas por defecto para comparaciones generales.
        metricas = ["Min", "Gls", "Ast", "xG", "npxG", "xAG", "xA", "PrgC", "PrgP", "PrgR", "SCA", "SCA90"]
        comparacion_df = comparar_jugadores(jugador_1, jugador_2, metricas=metricas)
        datos_csv = dataframe_a_texto(comparacion_df, f"Comparacion entre {jugador_1} y {jugador_2}")
        return {"datos_csv": datos_csv}

    # Caso conceptual: no hace falta consultar el CSV.
    return {"datos_csv": "La pregunta es conceptual; no se ha realizado consulta tabular al CSV."}


# Nodo 4: genera la respuesta final con Gemini usando memoria.
def nodo_generar_respuesta(state: AgentState):
    # Construimos el prompt con historial, pregunta actual, contexto RAG y datos CSV.
    prompt = f"""
{SYSTEM_PROMPT}

Historial reciente de la conversacion:
{state["historial"]}

Pregunta actual del usuario:
{state["pregunta"]}

Tipo de consulta detectada:
{state["tipo_consulta"]}

Contexto recuperado desde ChromaDB:
{state["contexto_rag"]}

Datos calculados desde el CSV con pandas:
{state["datos_csv"]}

Instrucciones:
- Usa el historial para entender referencias de seguimiento.
- Usa siempre el contexto recuperado desde ChromaDB para explicar metricas.
- Si hay datos del CSV, integrarlos en la respuesta.
- No inventes valores que no aparezcan en el contexto o en los datos del CSV.
- Si la pregunta depende del historial, explica brevemente a que elemento anterior te refieres.
- Cita las fuentes conceptuales indicando PDF y pagina cuando sea util.
""".strip()

    # Pedimos a Gemini la respuesta final.
    respuesta = llm.invoke(prompt)

    # Devolvemos la respuesta.
    return {"respuesta": respuesta.content}

<a id="sec-17-4"></a>
### ***17.4 Construccion del grafo final con MemorySaver***

Compilamos el grafo con `MemorySaver`, igual que en clase. El `thread_id` que pasaremos al invocar permite separar conversaciones o usuarios.


In [ ]:
# Creamos un grafo cuyo estado es AgentState.
graph_builder = StateGraph(AgentState)

# A?adimos los nodos.
graph_builder.add_node("clasificar_pregunta", nodo_clasificar_pregunta)
graph_builder.add_node("recuperar_contexto", nodo_recuperar_contexto)
graph_builder.add_node("consultar_csv", nodo_consultar_csv)
graph_builder.add_node("generar_respuesta", nodo_generar_respuesta)

# Definimos el flujo lineal del agente desde START hasta END.
graph_builder.add_edge(START, "clasificar_pregunta")
graph_builder.add_edge("clasificar_pregunta", "recuperar_contexto")
graph_builder.add_edge("recuperar_contexto", "consultar_csv")
graph_builder.add_edge("consultar_csv", "generar_respuesta")
graph_builder.add_edge("generar_respuesta", END)

# Creamos memoria por checkpoint para LangGraph.
memory = MemorySaver()

# Compilamos el grafo final con memoria.
agente_experto = graph_builder.compile(checkpointer=memory)

print("Agente experto con LangGraph y memoria compilado correctamente")

<a id="sec-17-5"></a>
### ***17.5 Visualizacion del grafo final***

Visualizamos el flujo del agente para comprobar que LangGraph conecta correctamente los nodos desde `START` hasta `END`.


In [ ]:
# Image y display permiten mostrar la imagen del grafo dentro del notebook.
from IPython.display import Image, display

# Dibujamos el grafo compilado en formato Mermaid PNG.
display(Image(agente_experto.get_graph().draw_mermaid_png()))

<a id="sec-17-6"></a>
### ***17.6 Funcion de chat con memoria***

Esta funcion guarda cada pregunta como `HumanMessage` y cada respuesta como `AIMessage`. Ademas, invoca el grafo con un `thread_id` para que `MemorySaver` pueda mantener checkpoints de la conversacion.


In [ ]:
# Historial simple de conversacion usando mensajes de LangChain.
# Usamos HumanMessage y AIMessage porque es el formato habitual visto en clase.
historial_conversacion = []


# Convierte el historial de mensajes en texto para incluirlo en el estado del grafo.
def formatear_historial(historial, max_mensajes=8):
    # Nos quedamos solo con los ultimos mensajes para no hacer prompts enormes.
    mensajes_recientes = historial[-max_mensajes:]

    # Si no hay historial, devolvemos mensaje claro.
    if not mensajes_recientes:
        return "No hay historial previo."

    # Convertimos cada mensaje a texto diferenciando usuario y asistente.
    lineas = []
    for mensaje in mensajes_recientes:
        if isinstance(mensaje, HumanMessage):
            lineas.append(f"Usuario: {mensaje.content}")
        elif isinstance(mensaje, AIMessage):
            lineas.append(f"Asistente: {mensaje.content}")
        elif isinstance(mensaje, SystemMessage):
            lineas.append(f"Sistema: {mensaje.content}")

    return "".join(lineas)


# Ejecuta el agente final con memoria y actualiza el historial.
def preguntar_agente(pregunta, thread_id="demo_usuario"):
    # Formateamos historial antes de responder.
    historial_texto = formatear_historial(historial_conversacion)

    # Creamos estado inicial. Los campos se iran completando en los nodos.
    estado_inicial = {
        "pregunta": pregunta,
        "historial": historial_texto,
        "tipo_consulta": None,
        "contexto_rag": None,
        "fuentes_rag": None,
        "datos_csv": None,
        "respuesta": None,
    }

    # Config permite a MemorySaver guardar el estado por conversacion o usuario.
    config = {"configurable": {"thread_id": thread_id}}

    # Ejecutamos el grafo final.
    resultado = agente_experto.invoke(estado_inicial, config=config)

    # Guardamos el mensaje humano y la respuesta del asistente en formato LangChain.
    historial_conversacion.append(HumanMessage(content=pregunta))
    historial_conversacion.append(AIMessage(content=resultado["respuesta"]))

    # Devolvemos estado final completo.
    return resultado

<a id="sec-17-7"></a>
### ***17.7 Pruebas del agente final***

Probamos tres tipos de preguntas: conceptual, ranking y comparacion. En todas ellas el agente recupera contexto desde ChromaDB.


In [ ]:
# Prueba 1: pregunta conceptual, usa ChromaDB y Gemini.
resultado_agente_conceptual = preguntar_agente("Que significa xG y como se interpreta en scouting?")
print(resultado_agente_conceptual["respuesta"])

In [ ]:
# Prueba 2: pregunta de ranking, usa ChromaDB + pandas + Gemini.
resultado_agente_ranking = preguntar_agente("Cuales son los jugadores con mas xG y que significa esta metrica?")
print(resultado_agente_ranking["respuesta"])

In [ ]:
# Prueba 3: pregunta de comparacion, usa ChromaDB + pandas + Gemini.
resultado_agente_comparacion = preguntar_agente("Compara a Mohamed Salah y Raphinha usando xG, xAG, PrgC y SCA")
print(resultado_agente_comparacion["respuesta"])

<a id="sec-17-7-1"></a>
### ***17.7.1 Prueba con lenguaje natural en espa?ol***

Esta prueba valida que el agente traduce expresiones como "delantero" a `FW` y "precision de tiro" a `SoT%` antes de consultar pandas.

In [ ]:
# Prueba adicional: lenguaje natural en espa?ol que requiere mapeo a columnas del CSV.
resultado_agente_precision = preguntar_agente("Que delantero tiene mayor precision de tiro?")
print(resultado_agente_precision["respuesta"])

<a id="sec-17-7-2"></a>
### ***17.7.2 Pruebas de mapeo ampliado***

Estas pruebas no llaman a Gemini. Solo comprueban que el agente traduce expresiones naturales a columnas del CSV antes de ejecutar pandas.

In [ ]:
# Pruebas rapidas del mapeo de lenguaje natural.
pruebas_mapeo_lenguaje_natural = [
    "Cual es el jugador con mas pases?",
    "Que delantero tiene mayor precision de tiro?",
    "Que portero tiene mejor porcentaje de paradas?",
    "Que jugador tiene mas pases progresivos?",
    "Cuales son los jugadores mas jovenes?",
]

for pregunta in pruebas_mapeo_lenguaje_natural:
    print("Pregunta:", pregunta)
    print("Metrica detectada:", detectar_metrica_en_pregunta(pregunta))
    print("Posicion detectada:", detectar_posicion_en_pregunta(pregunta))
    print("Liga detectada:", detectar_liga_en_pregunta(pregunta))
    print("Orden ascendente:", detectar_orden_ascendente(pregunta))
    print("-" * 80)

<a id="sec-17-8"></a>
### ***17.8 Ejemplo de memoria conversacional***

Este ejemplo demuestra memoria con una pregunta de seguimiento. La segunda pregunta usa "esa metrica", que hace referencia a `PSxG+/-` mencionada en el turno anterior.


In [ ]:
# Reiniciamos la memoria para que el ejemplo sea limpio.
historial_conversacion.clear()

# Primera pregunta: introduce la metrica PSxG+/-.
respuesta_memoria_1 = preguntar_agente("Explica que significa PSxG+/- para evaluar porteros")
print(respuesta_memoria_1["respuesta"])

In [ ]:
# Segunda pregunta: depende del contexto anterior mediante "esa metrica".
respuesta_memoria_2 = preguntar_agente("Y si esa metrica es positiva, que interpretacion tendria?")
print(respuesta_memoria_2["respuesta"])

<a id="sec-17-9"></a>
### ***17.9 Verificacion de memoria y uso de ChromaDB***

Estas celdas ayudan a demostrar que el agente conserva mensajes y recupera fuentes desde la base vectorial.


In [ ]:
# Mostramos los mensajes guardados en memoria.
for i, mensaje in enumerate(historial_conversacion, start=1):
    print(f"Mensaje {i}")
    print("Tipo:", type(mensaje).__name__)
    print("Contenido:", mensaje.content[:500], "...")
    print()

In [ ]:
# Mostramos las fuentes recuperadas por ChromaDB en la ultima pregunta de comparacion.
resultado_agente_comparacion["fuentes_rag"]

<a id="sec-18"></a>
## ***18. Interaccion en el notebook y ejemplos documentados***

Esta seccion cumple el Paso 6 del enunciado. Incluye una celda interactiva para conversar con el agente y cinco preguntas de ejemplo con sus respuestas documentadas.

<a id="sec-18-1"></a>
### ***18.1 Celda interactiva de chat***

La siguiente funcion permite hablar con el agente desde el notebook. Escribe preguntas una a una y usa `salir` para terminar la conversacion.

In [ ]:
# Funcion interactiva para conversar con el agente desde el notebook.
def chat_interactivo(thread_id="chat_notebook"):
    # Mensaje inicial para el usuario.
    print("Chat iniciado. Escribe 'salir' para terminar.")

    # Bucle de conversacion.
    while True:
        # Leemos pregunta del usuario.
        pregunta = input("Usuario: ")

        # Permitimos salir del chat.
        if pregunta.lower().strip() in ["salir", "exit", "quit"]:
            print("Chat finalizado.")
            break

        # Si el usuario envia texto vacio, pedimos otra entrada.
        if not pregunta.strip():
            print("Escribe una pregunta o 'salir' para terminar.")
            continue

        # Ejecutamos el agente final con memoria.
        resultado = preguntar_agente(pregunta, thread_id=thread_id)

        # Mostramos la respuesta inmediata.
        print("Asistente:")
        print(resultado["respuesta"])
        print("" + "-" * 100 + "")

In [ ]:
# Descomenta esta linea para iniciar una conversacion manual en el notebook.
chat_interactivo()

<a id="sec-18-2"></a>
### ***18.2 Cinco preguntas de ejemplo documentadas***

Estas preguntas cubren los casos principales del proyecto:

1. Pregunta conceptual con RAG.
2. Ranking con pandas + RAG.
3. Comparacion con pandas + RAG.
4. Pregunta sobre porteros con RAG.
5. Pregunta de seguimiento para demostrar memoria.

In [ ]:
# Reiniciamos la memoria para que los ejemplos sean reproducibles.
historial_conversacion.clear()

# Definimos cinco preguntas de evaluacion.
preguntas_ejemplo = [
    "Que significa xG y como se interpreta en scouting?",
    "Cuales son los jugadores con mas xG y que significa esta metrica?",
    "Compara a Mohamed Salah y Raphinha usando xG, xAG, PrgC y SCA",
    "Que significa PSxG+/- y como se usa para evaluar porteros?",
    "Y si esa metrica es positiva, que interpretacion tendria?",
]

# Ejecutamos las preguntas una a una con el mismo thread_id para conservar memoria.
respuestas_ejemplo = []
for i, pregunta in enumerate(preguntas_ejemplo, start=1):
    print("=" * 100)
    print(f"Pregunta {i}: {pregunta}")
    print("=" * 100)

    # Ejecutamos el agente.
    resultado = preguntar_agente(pregunta, thread_id="ejemplos_documentados")

    # Guardamos resultado para inspeccion posterior.
    respuestas_ejemplo.append(resultado)

    # Mostramos respuesta.
    print(resultado["respuesta"])
    print()

<a id="sec-18-3"></a>
### ***18.3 Documentacion de fuentes usadas en los ejemplos***

Para demostrar que el agente usa la base vectorial, mostramos las fuentes recuperadas desde ChromaDB en cada pregunta.

In [ ]:
# Mostramos tipo de consulta y fuentes RAG usadas en cada ejemplo.
for i, resultado in enumerate(respuestas_ejemplo, start=1):
    print("=" * 100)
    print(f"Ejemplo {i}")
    print("Tipo de consulta:", resultado.get("tipo_consulta"))
    print("Fuentes recuperadas desde ChromaDB:")

    for fuente in resultado.get("fuentes_rag", []):
        print("-", fuente)

    print()

<a id="sec-18-4"></a>
### ***18.4 Verificacion final de memoria***

Mostramos los ultimos mensajes guardados como `HumanMessage` y `AIMessage`. Esto evidencia que el agente conserva historial conversacional.

In [ ]:
# Mostramos los mensajes guardados en memoria despues de los ejemplos.
for i, mensaje in enumerate(historial_conversacion[-10:], start=1):
    print(f"Mensaje {i}")
    print("Tipo:", type(mensaje).__name__)
    print("Contenido:", mensaje.content[:500], "...")
    print()

<a id="resultado-fase"></a>
## ***Resultado de esta fase***

En este punto ya tenemos:

- 3 PDFs de metricas cargados como documentos de conocimiento.
- PDFs divididos en chunks semanticos.
- Embeddings creados con Gemini.
- Base vectorial persistente en ChromaDB.
- Retriever probado antes de conectar el agente.
- RAG basico con Gemini usando contexto recuperado desde los PDFs.
- Consultas pandas sobre el CSV completo.
- Respuestas hibridas que combinan ChromaDB + pandas + Gemini.
- Un unico agente final con LangGraph, MemorySaver y memoria conversacional.

A partir de aqui, solo falta preparar una celda interactiva de chat, incluir 5 preguntas de ejemplo documentadas y completar el README.
